In [1]:
import _referAsMain

import math
import time
import random
import sys
import numpy
from holo.profilers import Profiler
from holo.prettyFormats import prettyPrint, prettyTime
print(sys.version_info)

import torch
from torch.utils.data import DataLoader

from paths_cfg import TOKENIZER_SAVE_DIRECTORY, OUR_DATASET_DIRECTORY
from dataset import svg_dataset
import tokenizer_pfe.tokenizer_project as tokenizerLib
import metrics.metrics

from LLM.model import Model
import LLM.nanochat.gpt as nanoChatModel
from LLM.nanochat.common import compute_init, autodetect_device_type

added '/home/holo/dev/PFE_LLM_art_generation' to import paths
sys.version_info(major=3, minor=10, micro=19, releaselevel='final', serial=0)


In [2]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

Torch version: 2.9.1+cu128
CUDA available: True
CUDA version: 12.8
Device count: 1


In [3]:
tokenizer_path = TOKENIZER_SAVE_DIRECTORY.joinpath("our_tokenizer")
tokenizer = tokenizerLib.Tokenizer.load(tokenizer_path)

loading the tokenizer from: /home/holo/dev/PFE_LLM_art_generation/tokenizer_save/our_tokenizer.json


In [4]:
dataset = svg_dataset.SVGDataset(
    OUR_DATASET_DIRECTORY, context_size=4096,
    tokenizer=tokenizer.encode, decoder=tokenizer.decode)

In [5]:
device_type = "cuda"
device_type = (autodetect_device_type() if device_type == "" else device_type)
ddp, ddp_rank, ddp_local_rank, ddp_world_size, device = \
    compute_init(device_type)

2026-03-08 19:37:47,262 - LLM.nanochat.common - INFO - Distributed world size: 1


In [99]:
aspect_ratio = 10.5
head_dim = 128
max_seq_len = 2048*8
window_pattern = "SSSL"
vocab_size = 2048


def build_model_meta(depth):
    """Build a model on meta device for a given depth (shapes/dtypes only, no data)."""
    # Model dim is nudged up to nearest multiple of head_dim for clean division
    # (FA3 requires head_dim divisible by 8, and this guarantees head_dim == args.head_dim exactly)
    base_dim = depth * aspect_ratio
    model_dim = int(((base_dim + head_dim - 1) // head_dim) * head_dim)
    num_heads = int(model_dim // head_dim)
    print(f"num heads: {model_dim / head_dim}")
    config = nanoChatModel.GPTConfig(
        sequence_len=max_seq_len, vocab_size=vocab_size,
        n_layer=depth, n_head=num_heads, n_kv_head=num_heads, n_embd=model_dim,
        window_pattern=window_pattern)
    print(config)
    with torch.device("meta"):
        model_meta = nanoChatModel.GPT(config)
    return model_meta

model = Model()

model.llm = build_model_meta(6)
# 2) All tensors get storage on target device but with uninitialized (garbage) data
model.llm.to_empty(device=device)
model.llm.init_weights()  # 3) All tensors get initialized
model.optimizer = model.llm.setup_optimizer()

params = model.llm.num_scaling_params()
print(params.items())
params_Embed = (params['wte'] + params['value_embeds'])
print(f"{params['total']:_d} params "
      f"(with embeding: {params_Embed:_d} | "
      f"last layer: {params['lm_head']:_d} | "
      f"transformer: {params['transformer_matrices']:_d})")
model.llm = model.llm.bfloat16()

# the inputs to model will never change shape so dynamic=False is safe
model.llm = torch.compile(model.llm, dynamic=False) # type: ignore
# model.eval();


num heads: 1.0
GPTConfig(sequence_len=16384, vocab_size=2048, n_layer=6, n_head=1, n_kv_head=1, n_embd=128, window_pattern='SSSL')
Scaling the LR for the AdamW parameters ∝1/√(128/768) = 2.449490
dict_items([('wte', 262144), ('value_embeds', 786432), ('lm_head', 262144), ('transformer_matrices', 1179744), ('scalars', 12), ('total', 2490476)])
2_490_476 params (with embeding: 1_048_576 | last layer: 262_144 | transformer: 1_179_744)


In [100]:
import importlib
import metrics.metrics
_ = importlib.reload(metrics.metrics)
import metrics.metrics

In [89]:
N = 8
testDevice = torch.device("cuda:0")

prof = Profiler(["iterDataloader", "split", "target", "logits", "print"])

dataloader = DataLoader(dataset, batch_size=N, shuffle=True, num_workers=0)
accum = metrics.metrics.MetricsAccumulator(usage="train", topK=5)


print(f"there are {len(dataloader)} batch to process")
it = iter(enumerate(dataloader))
i = 0
t0 = time.perf_counter()
while True:
    with prof.mesure("iterDataloader"):
        try: i, datas = next(it)
        except StopIteration:
            break
    
    with prof.mesure("print"):
        if i % 5 == 0:
            print(f"\rdoing: {i=}", end="", flush=True)
            
    with prof.mesure("split"):
        tokens: torch.Tensor = datas["tokens"].to(torch.int64)
        assert isinstance(tokens, torch.Tensor)
        dtIndexes: list[int] = datas["datasetIndex"].tolist()
        svgIndex: list[int] = datas["svgIndex"].tolist()
        chunckIndex: list[int] = datas["chunckIndex"].tolist()
        nbChars: int = sum([len(dataset.chunks[i].text) for i in dtIndexes])

    with prof.mesure("target"):
        if True:
            target = tokens
        else: target = torch.tensor(list(range(256)), dtype=torch.int64).view(1, -1)
        target = target.to(testDevice)

    with prof.mesure("logits"):
        batchSize, seq_len = target.shape
        vocab_size = tokenizer.get_vocab_size()
        if True:
            logits = torch.full((batchSize, seq_len, vocab_size), -10.0, device=testDevice)
            mask = (target != svg_dataset.IGNORE_INDEX)
            b, t = mask.nonzero(as_tuple=True)
            logits[b, t, target[b, t]] = 0.0
            logits += 3.0 * torch.randn(batchSize, seq_len, vocab_size, device=testDevice)
        else: logits = torch.randn(batchSize, seq_len, vocab_size, device=testDevice)# * 0.001

    # print(f"logits of shape: {logits.shape}[{logits.dtype}]")
    # print(f"target of shape: {target.shape}[{target.dtype}]")
    
    accum.batch_logits_metrics(logits, target, totalNbChars=nbChars)
    
    if i*N > 50000:
        break

torch.cuda.empty_cache()
t1 = time.perf_counter()
print()
print(f"performed: {i} batch ({i*N} chuncks) in {prettyTime(t1-t0)}")
print(f" -> {i/(t1-t0):.2f} batch/sec | {i*N/(t1-t0):.2f} chuncks/sec")

print()
res = accum.get_metrics()
prettyPrint(prof.pretty_totalTimes())
prettyPrint(accum._prof.pretty_totalTimes())

there are 1725 batch to process
doing: i=1720
performed: 1724 batch (13792 chuncks) in 3.561 sec
 -> 484.17 batch/sec | 3873.37 chuncks/sec

{
    iterDataloader: 254.771 ms,
    split: 87.782 ms,
    target: 1.910 sec,
    logits: 1.011 sec,
    print: 170.969 ms
}
{loss1: 58.849 ms}


In [90]:
prettyPrint(res)

{
    CE_train: -0.0,
    CE2_train: -0.0,
    PPL_train: 1.0,
    PPL2_train: 1.0,
    BPC_train: -0.0,
    ENTROPY_train: -0.0,
    LOGITS_SD_train: -0.0,
    TOP-1_train: -0.0,
    TOP-5_train: -0.0
}


In [91]:
accum = metrics.metrics.MetricsAccumulator(usage="train", topK=5)
t1 = time.perf_counter()
accum.batch_logits_metrics(logits, target, totalNbChars=int(1.97*seq_len*batchSize))
t2 = time.perf_counter()
prettyPrint(accum.get_metrics())
prettyPrint(accum._prof.pretty_totalTimes())
tt = sum(accum._prof.totalTimes().values())
print(prettyTime(tt), tt/(t2-t1), batchSize/tt)

{
    CE_train: -0.0,
    CE2_train: -0.0,
    PPL_train: 1.0,
    PPL2_train: 1.0,
    BPC_train: -0.0,
    ENTROPY_train: -0.0,
    LOGITS_SD_train: -0.0,
    TOP-1_train: -0.0,
    TOP-5_train: -0.0
}
{loss1: 0.988 ms}
0.988 ms 0.9338772818758772 1012.2738212698232


In [113]:
metrics.metrics.get_learning_rates(model)

{'lr_lm_head': 0.009797958971132713,
 'lr_embedding': 0.4898979485566357,
 'lr_value_embeds': 0.4898979485566357,
 'lr_residuals': 0.005,
 'lr_x0': 0.5,
 'lr_transformers_grp_0': 0.02,
 'lr_transformers_grp_1': 0.02,
 'lr_transformers_grp_2': 0.02,
 'lr_transformers_grp_3': 0.02}

In [ ]:
metrics.metrics.get_learning_rates(model)

{'lr_lm_head': 0.009797958971132713,
 'lr_embedding': 0.4898979485566357,
 'lr_value_embeds': 0.4898979485566357,
 'lr_residuals': 0.005,
 'lr_x0': 0.5,
 'lr_transformers_grp_0': 0.02,
 'lr_transformers_grp_1': 0.02,
 'lr_transformers_grp_2': 0.02,
 'lr_transformers_grp_3': 0.02}

In [ ]:
N = 16
testDevice = torch.device("cuda:0")

prof = Profiler([
    "print", "iterDataloader", "split", "tokens",
    "forward", "metrics&loss", "zero_grad", "backward", "step"])

dataloader = DataLoader(dataset, batch_size=N, shuffle=True, num_workers=0)
accum = metrics.metrics.MetricsAccumulator(usage="train", topK=5)


print(f"there are {len(dataloader)} batch to process")
it = iter(enumerate(dataloader, start=0))
i = 0
t0 = time.perf_counter()
while True:
    torch.cuda.empty_cache()
    with prof.mesure("iterDataloader"):
        try: i, datas = next(it)
        except StopIteration:
            break
    
    with prof.mesure("print"):
        if i % 5 == 0:
            print(f"\rdoing: {i=}", end="", flush=True)
            
    with prof.mesure("split"):
        tokens: torch.Tensor = datas["tokens"].to(torch.int64)
        assert isinstance(tokens, torch.Tensor)
        dtIndexes: list[int] = datas["datasetIndex"].tolist()
        svgIndex: list[int] = datas["svgIndex"].tolist()
        chunckIndex: list[int] = datas["chunckIndex"].tolist()
        nbChars: int = sum([len(dataset.chunks[i].text) for i in dtIndexes])

    with prof.mesure("tokens"):
        tokens = tokens.to(testDevice)

    with prof.mesure("forward"):
        logits = model.llm.forward(idx=tokens, targets=None)
        
    # print(f"tokens of shape: {tokens.shape}[{tokens.dtype}]")
    # print(f"logits of shape: {logits.shape}[{logits.dtype}]")
    
    with prof.mesure("metrics&loss"):
        loss = accum.batch_logits_metrics(logits, tokens, totalNbChars=nbChars)
    
    with prof.mesure("zero_grad"):
        model.optimizer.zero_grad()
    with prof.mesure("backward"):
        loss.backward()
    with prof.mesure("step"):
        model.optimizer.step()
    
    if i*N >= 5000:
        break

torch.cuda.empty_cache()
t1 = time.perf_counter()
print()
print(f"performed: {i} batch ({i*N} chuncks) in {prettyTime(t1-t0)}")
print(f" -> {i/(t1-t0):.2f} batch/sec | {i*N/(t1-t0):.2f} chuncks/sec")
res = accum.get_metrics()
prettyPrint(prof.pretty_totalTimes())
prettyPrint(accum._prof.pretty_totalTimes())
prettyPrint(res)


there are 863 batch to process
doing: i=310
performed: 313 batch (5008 chuncks) in 1 min 0.5 sec
 -> 5.17 batch/sec | 82.75 chuncks/sec
{
    print: 41.078 ms,
    iterDataloader: 109.098 ms,
    split: 64.224 ms,
    tokens: 47.053 ms,
    forward: 12.955 sec,
    metrics&loss: 24.000 sec,
    zero_grad: 49.505 ms,
    backward: 1.592 sec,
    step: 1.488 sec
}
{
    loss1: 22.900 ms,
    loss2: 922.052 ms,
    filter: 8.894 sec,
    CE_related: 3.412 sec,
    entropy: 8.244 sec,
    SD: 301.793 ms,
    topK: 147.841 ms,
    accuacy: 2.018 sec
}
{
    CE_train: 0.0004963519240083425,
    CE2_train: 0.0004741914707500883,
    PPL_train: 1.0004964751270078,
    PPL2_train: 1.0004743039172985,
    BPC_train: 0.0003628412864993868,
    ENTROPY_train: 0.0059185421065941005,
    LOGITS_SD_train: 0.9937233674129241,
    TOP-1_train: 0.9999894755077688,
    TOP-5_train: 0.9999999020977467
}


In [114]:
prettyPrint(res)

{
    CE_train: 0.0004963519240083425,
    CE2_train: 0.0004741914707500883,
    PPL_train: 1.0004964751270078,
    PPL2_train: 1.0004743039172985,
    BPC_train: 0.0003628412864993868,
    ENTROPY_train: 0.0059185421065941005,
    LOGITS_SD_train: 0.9937233674129241,
    TOP-1_train: 0.9999894755077688,
    TOP-5_train: 0.9999999020977467
}


# stats when testing

there are 13793 batch to process
doing: i=500
performed: 500 batch (500 chuncks) in 18.993 sec
 -> 26.38 batch/sec | 26.38 chuncks/sec
{
    print: 69.003 ms,
    iterDataloader: 100.535 ms,
    split: 28.354 ms,
    tokens: 86.609 ms,
    forward: 7.559 sec,
    metrics&loss: 2.347 sec,
    zero_grad: 57.810 ms,
    backward: 2.479 sec,
    step: 2.458 sec
}

there are 6897 batch to process
doing: i=250
performed: 250 batch (500 chuncks) in 9.353 sec
 -> 26.73 batch/sec | 53.46 chuncks/sec
{
    print: 37.736 ms,
    iterDataloader: 58.378 ms,
    split: 15.162 ms,
    tokens: 41.625 ms,
    forward: 1.953 sec,
    metrics&loss: 3.115 sec,
    zero_grad: 31.538 ms,
    backward: 1.206 sec,
    step: 1.501 sec
}

there are 3449 batch to process
doing: i=125
performed: 125 batch (500 chuncks) in 7.436 sec
 -> 16.81 batch/sec | 67.24 chuncks/sec
{
    print: 18.826 ms,
    iterDataloader: 35.273 ms,
    split: 7.954 ms,
    tokens: 15.757 ms,
    forward: 1.688 sec,
    metrics&loss: 2.804 sec,
    zero_grad: 18.196 ms,
    backward: 603.035 ms,
    step: 697.346 ms
}

there are 1725 batch to process
doing: i=60
performed: 63 batch (504 chuncks) in 6.883 sec
 -> 9.15 batch/sec | 73.22 chuncks/sec
{
    print: 11.293 ms,
    iterDataloader: 28.092 ms,
    split: 6.280 ms,
    tokens: 9.228 ms,
    forward: 1.543 sec,
    metrics&loss: 2.764 sec,
    zero_grad: 11.558 ms,
    backward: 302.809 ms,
    step: 381.092 ms
}

there are 863 batch to process
doing: i=30
performed: 32 batch (512 chuncks) in 6.476 sec
 -> 4.94 batch/sec | 79.07 chuncks/sec
{
    print: 5.502 ms,
    iterDataloader: 18.805 ms,
    split: 7.956 ms,
    tokens: 5.724 ms,
    forward: 1.413 sec,
    metrics&loss: 2.599 sec,
    zero_grad: 7.059 ms,
    backward: 164.066 ms,
    step: 168.861 ms
}

there are 863 batch to process
doing: i=30
performed: 32 batch (512 chuncks) in 4.626 sec
 -> 6.92 batch/sec | 110.68 chuncks/sec
{
    print: 4.346 ms,
    iterDataloader: 12.407 ms,
    split: 6.354 ms,
    tokens: 5.839 ms,
    forward: 1.270 sec,
    metrics&loss: 3.479 ms,
    zero_grad: 3.922 ms,
    backward: 354.835 ms,
    step: 148.873 ms
}